# Optimal Model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import math

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
from mlxtend.evaluate import GroupTimeSeriesSplit
from mlxtend.evaluate.time_series import plot_splits
from sklearn.metrics import mean_squared_error, r2_score

import keras

import tensorflow as tf
from tensorflow.keras.layers import MaxPooling1D
import datetime

%load_ext tensorboard


In [ ]:
# GPU Availability Check
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# List all physical devices
physical_devices = tf.config.list_physical_devices()
print("\n📱 All Physical Devices:")
for device in physical_devices:
    print(f"  {device}")

# Check if TensorFlow is built with CUDA support
print(f"\n🔧 CUDA Support: {tf.test.is_built_with_cuda()}")


In [ ]:
def set_reproducible(seed=12345):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
dps1200 = pd.read_csv("../../data/dps1200.csv")
dps1200 = dps1200.sort_values("year", ascending=True)
dps1200.head()

In [ ]:
features = dps1200.iloc[:, 4:].values
labels = dps1200.iloc[:, 0].values

In [ ]:
def convertToDecade(y:int) -> int: 
    return int(str(y)[:3])

def calculate_sample_weights(y_train):

    decades = [convertToDecade(year) for year in y_train]

    unique_decades, counts = np.unique(decades, return_counts=True)
    total_samples = len(y_train)
    
    weights = {}
    for decade, count in zip(unique_decades, counts):
        weights[decade] = 1 - count/total_samples

    sample_weights = []
    for year in y_train:
        sample_weights.append(weights[convertToDecade(year)])
        
    return np.array(sample_weights)            


First redifine the model into a function where the hyperparameters are inputs. This function evaluates the model and ouptups a desired score/metric.

In [ ]:
def build_model_better_old(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K, C3_S, input_dim):

    activation1='relu'
    activation2='linear'
    
    model = keras.Sequential()
    model.add(keras.layers.Input((input_dim, 1)))
    # model.add(keras.layers.GaussianNoise(0.0001))  # better without

    model.add(keras.layers.Conv1D(C1_K, (C1_S), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(C2_K, (C2_S), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (50), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(C3_K, (C3_S), padding='valid', activation=activation1))
    # model.add(MaxPooling1D(pool_size=2))  # better without

    model.add(keras.layers.Flatten())
    # model.add(keras.layers.Dropout(DropoutR))  # better without
    
    model.add(keras.layers.Dense(DenseN, activation=activation1))
    model.add(keras.layers.Dense(DenseN, activation=activation1))
    model.add(keras.layers.Dense(DenseN, activation=activation1))
    model.add(keras.layers.Dense(1, activation=activation2))

    model.compile(loss=tf.keras.losses.Huber(), optimizer=keras.optimizers.Adam(learning_rate=0.001), metrics=['mean_absolute_error'])
    
    return model


In [ ]:
def build_model(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K, C3_S, input_dim):
    activation1='relu'
    activation2='linear'
    
    model = keras.Sequential()
    model.add(keras.layers.Input((input_dim, 1)))

    model.add(keras.layers.Conv1D(55, (45), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(43, (36), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (50), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (50), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (50), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (50), padding='valid', activation=activation1))
    model.add(keras.layers.Conv1D(30, (1), padding='valid', activation=activation1))

    model.add(keras.layers.Flatten())
    
    model.add(keras.layers.Dense(351, activation=activation1))
    model.add(keras.layers.Dense(256, activation=activation1))
    model.add(keras.layers.Dense(128, activation=activation1))
    model.add(keras.layers.Dense(64, activation=activation1))
    model.add(keras.layers.Dense(32, activation=activation1))
    model.add(keras.layers.Dense(1, activation=activation2))

    model.compile(loss=tf.keras.losses.Huber(), optimizer=keras.optimizers.Adam(learning_rate=0.001), metrics=['mean_absolute_error'])

    # print model summary
    model.summary()

    return model

# Crossvalidation

Since NN training involves random sampling and weights initialization (in this case), it is usefull to use cross-validation.

In [ ]:
def recalculate_year(years):
    result = np.zeros_like(years)
    for i in range(len(years)):
        result[i] = math.exp(years[i])
    return result

In [ ]:
## Compute error metrics
def error_metrics(y_true_train, y_predicted_train, y_true_test, y_predicted_test):
    y_true_train = recalculate_year(y_true_train)
    y_predicted_train = recalculate_year(y_predicted_train)
    y_true_test = recalculate_year(y_true_test)
    y_predicted_test = recalculate_year(y_predicted_test)

    rmse_train = np.sqrt(mean_squared_error(y_true_train, y_predicted_train))
    rmse_test = np.sqrt(mean_squared_error(y_true_test, y_predicted_test))
    R2_train= r2_score(y_true_train, y_predicted_train)
    R2_test= r2_score(y_true_test, y_predicted_test)
    h = tf.keras.losses.Huber()
    hub_train = h(y_true_train, y_predicted_train).numpy()
    hub_test = h(y_true_test, y_predicted_test).numpy()
    
    print('*********** Benchmark results ***********\n')
    print(f"R2    (Train/Test) = {R2_train:.3f} / {R2_test:.3f}")
    print(f"RMSE  (Train/Test) = {rmse_train:.3f} / {rmse_test:.3f}")
    print(f"Huber (Train/Test) = {hub_train:.3f} / {hub_test:.3f}")

    return (rmse_train, rmse_test, R2_train, R2_test, hub_train, hub_test)

class ModelWithData:
    def __init__(self, history, train_x, train_label, train_predicted, test_x, test_label, test_predicted, score:float, iteration, model) -> None:
        self.history = history
        self.train_x = train_x
        self.train_label = train_label
        self.train_predicted = train_predicted
        self.test_x = test_x
        self.test_label = test_label
        self.test_predicted = test_predicted
        self.score = score
        self.iteration = iteration
        self.model = model

    def isBetter(self, otherScore: float) -> bool:
        return otherScore < 0 or (self.score >= 0 and self.score <= otherScore)

In [ ]:
def calculate_groups(years, years_in_group=100):
    groups = np.zeros(len(years))

    for i in range(len(years)):
        groups[i] = years[i] // years_in_group
    
    print("Groups:")
    print(groups)
    return groups

In [ ]:
class HarmonizedTimeSplit:
    def __init__(self, validation_portion=0.1, random_seed=12345):
        self._validation_portion = validation_portion
        self._random_seed = random_seed
        self._random = random.Random(random_seed)

    def split(self, X, y = None, groups = None):

        if groups is None:
            raise ValueError("The groups should be specified")
        
        self._random = random.Random(self._random_seed)
        group_dict = self._calculate_group_indices(groups)
        train_indices = []
        test_indices = []

        for group_start, group_end in group_dict.values():
            group_indices = []

            for index in range(group_start, group_end):
                group_indices.append(index)
            
            self._random.shuffle(group_indices)

            validation_end = math.floor(len(group_indices)*self._validation_portion)

            if validation_end <= 0:
                train_indices.extend(group_indices)
            else:
                test_indices.extend(group_indices[:validation_end])
                train_indices.extend(group_indices[validation_end:])

        yield np.asarray(sorted(train_indices)), np.asarray(sorted(test_indices))

        
    def _calculate_group_indices(self, groups) -> dict[str, tuple[int,int]]:
        group_dict = {}
        current_group = None

        for index in range(len(groups)):
            group = str(groups[index])

            if group in group_dict:
                if group != current_group:
                    raise ValueError(f"The groups should be specified in consequtive order. Found group '{group}' again at index {index}")
            else:
                if current_group is not None:
                    (current_group_start, current_group_end) = group_dict[current_group]
                    group_dict[current_group] = (current_group_start, index)
                
                current_group = group
                group_dict[group] = (index, len(groups))

        return group_dict

    def plot_splits(self, groups):
        for train_indices, test_indices in self.split(None, None, groups):
            self._plot_split(train_indices, test_indices, groups)
    
    def _plot_split(self, train_indices, test_indices, groups):
        group_dict = self._calculate_group_indices(groups)

        train_in_group = np.zeros(len(group_dict))
        test_in_group = np.zeros(len(group_dict))
        groups = sorted(group_dict.keys())

        for group_index, group_name in enumerate(groups):
            (group_start, group_end) = group_dict[group_name]
            train_in_group[group_index] = self._count_in_group(train_indices, group_start, group_end)
            test_in_group[group_index] = self._count_in_group(test_indices, group_start, group_end)

        x_values = np.arange(len(groups))
        width = 0.33

        rects = plt.bar(x_values, train_in_group, width, color='k', label='Train')
        plt.bar_label(rects, padding=3)
        rects = plt.bar(x_values + width, test_in_group, width, color='r', label='Test')
        plt.bar_label(rects, padding=3)

        # Labels, legend, and title
        plt.xlabel('Groups')
        plt.xticks(x_values + width/2, groups)
        plt.ylabel('Sample Count')
        plt.legend()
        plt.title('Distribution of Train vs Test in Split')

        # Show the plot
        plt.show()
    
    def _count_in_group(self, indices, group_start, group_end):
        count = 0
        for train_index in indices:
            if train_index >= group_start and train_index < group_end:
                count += 1
            elif train_index >= group_end:
                break

        return count


In [ ]:
hts = HarmonizedTimeSplit(validation_portion=0.1)
y = np.array(labels)
groups = calculate_groups(y, 100)
hts.plot_splits(groups)

In [ ]:
def evaluations_of_models(features, labels, DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K, C3_S, seed=12345, name="Anabell", epochs=100, years_in_group=100, validation_portion=0.01):

    batch_size = 45

    x = np.array(features, dtype=np.float32)  # Convert to float32 upfront
    y_int = np.array(labels)
    y = np.zeros(len(y_int), dtype=np.float32)  # Use float32
    
    groups = calculate_groups(y_int, years_in_group)
    input_dim = 410

    for i in range(len(y_int)):
        y[i] = math.log(y_int[i])

    print(y)
    
    # Ensure proper shape for CNN (add channel dimension if needed)
    if x.ndim == 2:
        x = x.reshape((x.shape[0], x.shape[1], 1))
    
    x_full_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_full_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    # generate model
    model = build_model(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K, C3_S, input_dim)

    # Define the number of folds for Cross-Validation
    hts = HarmonizedTimeSplit(validation_portion=validation_portion)
    hts.plot_splits(groups)

    scores_train = { "rmse": [], "r2": [], "huber": []}
    scores_test = { "rmse": [], "r2": [], "huber": []}

    bestMwd = ModelWithData(None, 0, 0, 0, 0, 0, 0, -1, 0, None)

    tensorboard_base_dir = "tensorboard_logs"
    time_string = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

    all_mwds = []
    
    # Iterate through the models
    i = 0
    for train_index, test_index in hts.split(x, y, groups):
        set_reproducible(seed)
        i = i + 1
        print(f'\n\n> Fold {i}')

        tensorboard_log_dir = f"{tensorboard_base_dir}/{name}/{time_string}/{seed}/fold_{i}"
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=tensorboard_log_dir)
        
        x_train_tensor = tf.gather(x_full_tensor, train_index)
        y_train_tensor = tf.gather(y_full_tensor, train_index)
        x_test_tensor = tf.gather(x_full_tensor, test_index)
        y_test_tensor = tf.gather(y_full_tensor, test_index)
        x_train = x[train_index]
        y_train = y[train_index]
        x_test = x[test_index]
        y_test = y[test_index]
        
        sample_weight = calculate_sample_weights(y_int[train_index])
        sample_weight_tensor = tf.convert_to_tensor(sample_weight, dtype=tf.float32)
        
        weight_dataset = tf.data.Dataset.from_tensor_slices(sample_weight_tensor)
        
        train_dataset_with_weights = tf.data.Dataset.zip((
            tf.data.Dataset.from_tensor_slices(x_train_tensor),
            tf.data.Dataset.from_tensor_slices(y_train_tensor), 
            weight_dataset
        ))
        
        train_dataset_with_weights = train_dataset_with_weights.shuffle(buffer_size=len(train_index))
        train_dataset_with_weights = train_dataset_with_weights.batch(batch_size)
        train_dataset_with_weights = train_dataset_with_weights.prefetch(tf.data.AUTOTUNE)

        val_dataset = tf.data.Dataset.from_tensor_slices((x_test_tensor, y_test_tensor))
        val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
        history = model.fit(train_dataset_with_weights,
                            epochs=epochs,
                            validation_data=val_dataset,
                            verbose=1,
                            callbacks=[tensorboard_callback])
        
        evaluation_results = model.evaluate(val_dataset, verbose=0)
        evaluation_labels = model.metrics_names

        for j in range(len(evaluation_results)):
            print(f'{evaluation_labels[j]}: {evaluation_results[j]}')

        train_pred_dataset = tf.data.Dataset.from_tensor_slices(x_train_tensor).batch(batch_size)
        test_pred_dataset = tf.data.Dataset.from_tensor_slices(x_test_tensor).batch(batch_size)
        train_pred = model.predict(train_pred_dataset, verbose=0)
        test_pred = model.predict(test_pred_dataset, verbose=0)

        train_pred = train_pred.flatten()  # Remove extra dimensions
        test_pred = test_pred.flatten()

        (rmse_train, rmse_test, r2_train, r2_test, huber_train, huber_test) = error_metrics(y_train, train_pred, y_test, test_pred)

        mwd = ModelWithData(history, x_train, y_train, train_pred, x_test, y_test, test_pred, rmse_train, i, model)

        all_mwds.append(mwd)

        if mwd.isBetter(bestMwd.score):
            bestMwd = mwd

        scores_train["rmse"].append(rmse_train)
        scores_train["r2"].append(r2_train)
        scores_train["huber"].append(huber_train)

        scores_test["rmse"].append(rmse_test)
        scores_test["r2"].append(r2_test)
        scores_test["huber"].append(huber_test)

    ## clear session 
    tf.keras.backend.clear_session()

    scores_train_mean = {}
    scores_test_mean = {}
    num_model = len(scores_train["rmse"])

    for metric in scores_train.keys():
        scores_train_mean[metric] = np.mean(scores_train[metric])
        scores_test_mean[metric] = np.mean(scores_test[metric])
        print(f'Train: {metric} (mean of {num_model} models)= {scores_train_mean[metric]} \nTest: {metric} (mean of {num_model} models)= {scores_test_mean[metric]}')
    
    return (bestMwd, all_mwds)

In [ ]:
# 3173, 6113
# Anabell: 2 Folds & 100 epochs
# Berta: 2 Folds & 1000 epochs
seeds = [3173] #, 6113]
all_mwds = []

for seed in seeds:
    print(f"Using seed {seed}")
    (bestMwd, all_mwds_of_run) = evaluations_of_models(features, labels, DenseN=351, DropoutR=0.0, C1_K=55, C1_S=45, C2_K=43, C2_S=36, C3_K=63, C3_S=1, seed=seed, name="Igor", epochs=1000, years_in_group=100, validation_portion=0.1)
    all_mwds.extend(all_mwds_of_run)

In [ ]:
def plot_model_performance(mwd: ModelWithData):
    # Create an array of x-values ranging from the minimum to maximum of the data
    x_values = np.linspace(min(labels), max(labels), 100)

    # Plot the line of equality (y=x)
    plt.plot(x_values, x_values, color='blue', linestyle='--', label='Y = X line')

    # Scatter plot for predicted values
    plt.scatter(mwd.train_label, mwd.train_predicted, c='k', label='Train Labels')
    plt.scatter(mwd.test_label, mwd.test_predicted, c='r', label='Test Labels')

    # Labels, legend, and title
    plt.xlabel('True Labels')
    plt.ylabel('Predicted Labels')
    plt.legend()
    plt.title('Scatter Plot of True vs Predicted Labels with Line of Equality')

    # Show the plot
    plt.show()

In [ ]:
for mdw in all_mwds:
    plot_model_performance(mdw)

In [ ]:
# Create an array of x-values ranging from the minimum to maximum of the data
x_values = np.linspace(min(labels), max(labels), 100)

# Plot the line of equality (y=x)
plt.plot(x_values, x_values, color='blue', linestyle='--', label='Y = X line')

# Scatter plot for predicted values
plt.scatter(bestMwd.train_label, bestMwd.train_predicted, c='k', label='Train Labels')
plt.scatter(bestMwd.test_label, bestMwd.test_predicted, c='r', label='Test Labels')

# Labels, legend, and title
plt.xlabel('True Labels')
plt.ylabel('Predicted Labels')
plt.legend()
plt.title('Scatter Plot of True vs Predicted Labels with Line of Equality')

# Show the plot
plt.show()

## Save the model

In [ ]:
bestMwd.model.save('dps1200sub_model.keras')

# Evaluation of the restored model

In [ ]:
evaluation_results = dps1200sub_model.evaluate(features, labels, verbose=0)
evaluation_labels = dps1200sub_model.metrics_names

for j in range(len(evaluation_results)):
    print(f'{evaluation_labels[j]}: {evaluation_results[j]}')


In [ ]:
def error_metrices_restored_model(y_true, y_predicted):
    rmse_train = np.sqrt(mean_squared_error(y_true, y_predicted))
    R2_train= r2_score(y_true, y_predicted)
    h = tf.keras.losses.Huber()
    hub_train = h(y_true, y_predicted).numpy()
    
    print('*********** Benchmark results ***********\n')
    print(f"R2    = {R2_train:.3f}")
    print(f"RMSE  = {rmse_train:.3f}")
    print(f"Huber = {hub_train:.3f}")

In [ ]:
restored_pred = dps1200sub_model.predict(features, verbose=0)

In [ ]:
error_metrices_restored_model(labels, restored_pred)